### Week04 김상우

In [ ]:
# 가중치 함수
import numpy as np


In [14]:
# drop out

import numpy as np

class Dropout:
    def __init__(self, dropout_ratio=0.5):
        if not 0 <= dropout_ratio < 1:
            raise ValueError("dropout_ratio는 0 이상 1 미만이어야 합니다.")
        self.dropout_ratio = dropout_ratio
        self.mask = None
        
    def forward(self, x, training=True):
        if training:
            keep_ratio = 1.0 - self.dropout_ratio
            self.mask = (np.random.rand(*x.shape) < keep_ratio) / keep_ratio
            return x * self.mask
        
        # Inverted dropout은 추론 시 별도의 크기 조정이 필요 없다.
        self.mask = None
        return x
        
    def backward(self, dout):
        if self.mask is None:
            raise RuntimeError("학습 모드 forward()를 먼저 실행해야 합니다.")
        return dout * self.mask

In [ ]:
# 배치 정규화 
import numpy as np
# 순전파
def batchnorm_forward(x, gamma, beta):
    
    # 평균 노드
    mean1_node = np.mean(x, axis = 0)
    
    # 차 노드
    sub_node = x - mean1_node
    
    # 제곱노드 (branch start)
    sqr_node = sub_node**2
    
    # 평균 노드
    mean2_node = np.mean(sqr_node, axis = 0)
    
    # 표준편차 노드
    std_node = np.sqrt(mean2_node + 1e-7)
    
    # 역수 노드 (branch end)
    inverse_node = 1. / std_node
    
    # 곱 노드
    mul1_node = sub_node * inverse_node
    
    # 곱노드 2
    mul2_node = mul1_node * gamma
    
    # 합 노드
    sum_node = mul2_node + beta
    
    cache  = (sub_node, inverse_node, mul1_node, gamma, std_node, mean2_node)
    return sum_node
    
    
# 역전파

def batchnorm_backward(dout, cache):
    
    sub_node, inverse_node, mul1_node, gamma, std_node, mean2_node = cache
    
    # 덧셈노드
    N = dout.shape[0]
    d_beta =  np.sum(dout, axis = 0)
    d_mul2 = dout
    
    # 곱노드 2
    d_gamma = np.sum(d_mul2 * mul1_node, axis = 0)
    d_mul1 = d_mul2 * gamma
    
    # 곱 노드 1
    d_inverse = np.sum(d_mul1 * sub_node, axis = 0)
    d_sub1 = d_mul1 * inverse_node
    
    # 역수 노드
    d_std = -d_inverse * (inverse_node ** 2)
    
    # 표준편차 노드
    d_var = d_std * 0.5 * inverse_node
    
    # mean2 노드
    d_sqr = d_var * np.ones_like(dout) / N
    
    # 제곱 노드
    d_sub2 = d_sqr * (2 * sub_node)
    # 차 노드
    d_sub = d_sub1 + d_sub2
    d_x1 = d_sub
    d_mean = -np.sum(d_sub, axis = 0)
    d_x2 = d_mean * np.ones_like(dout) / N
    
    # 평균 노드
    d_x = d_x1 + d_x2
    
    return d_x, d_gamma, d_beta
    

## 배치 정규화를 적용한 Neural Network

In [16]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def ReLU(x):
    return np.maximum(0, x)

def sigmoid_grad(output):

    return output * (1 - output)

def ReLU_grad(output):
    return (output >0).astype(float)

# 가중치 초기화
def He(input_size, output_size):
    return np.random.randn(input_size, output_size) * np.sqrt(2/input_size)

def Xavier(input_size, output_size):
    return ( np.random.randn(input_size, output_size) / np.sqrt(input_size))

class NeuralNetwork:
    def __init__(self, layer_size, activation, learning_rate, dropout_ratio=0.0, optimizer="sgd", momentum=0.9):
        self.layer_size = layer_size
        self.activation = activation
        if activation is sigmoid:
            self.activation_grad = sigmoid_grad
            self.weight_method = Xavier
        elif activation is ReLU:
            self.activation_grad = ReLU_grad
            self.weight_method = He
        else:
            print("Change activation function")
            
        self.learning_rate = learning_rate
        self.optimizer = optimizer.lower()
        self.momentum = momentum
        self.params = {}
        self.grads = {}
        
        self.A_values = []
        self.Z_values = []
        self.BN_chaches = [] # 배치 정규화
        
        self.output_value = None
        # 가중치 편향 생성
        
        for i in range(1, len(self.layer_size)):
            self.params[f"W{i}"] = self.weight_method(self.layer_size[i-1], self.layer_size[i])
            self.params[f"B{i}"] = np.zeros((1, self.layer_size[i]))
            
            if i < len(self.layer_size) - 1:
                self.params[f"Gamma{i}"] = np.ones((self.layer_size[i]))
                self.params[f"Beta{i}"] = np.zeros((self.layer_size[i]))
        
        # Momentum, AdaGrad
        self.velocity = {key: np.zeros_like(value) for key, value in self.params.items()}
        self.h = {key: np.zeros_like(value) for key, value in self.params.items()}
        
        # Dropout: 은닉층마다 서로 다른 mask를 저장해야 한다.
        self.dropout_ratio = dropout_ratio
        self.dropout_layers = {
            i: Dropout(dropout_ratio)
            for i in range(1, len(self.layer_size) - 1)
        }
        self.activation_values = []
    
    
    def batchnorm_forward(self, x, gamma, beta):
        mean1_node = np.mean(x, axis = 0)
        sub_node = x - mean1_node
        sqr_node = sub_node**2
        mean2_node = np.mean(sqr_node, axis = 0)
        std_node = np.sqrt(mean2_node + 1e-7)
        inverse_node = 1.0 / std_node
        mul1_node = sub_node * inverse_node
        mul2_node = mul1_node * gamma
        sum_node = mul2_node + beta
        cache  = (sub_node, inverse_node, mul1_node, gamma, std_node, mean2_node)
        return sum_node, cache
    
    
    def batchnorm_backward(self, dout, cache):
        
        sub_node, inverse_node, mul1_node, gamma, std_node, mean2_node = cache
        
        N = dout.shape[0]
        d_beta =  np.sum(dout, axis = 0)
        d_mul2 = dout
        
        d_gamma = np.sum(d_mul2 * mul1_node, axis = 0)
        d_mul1 = d_mul2 * gamma
        
        d_inverse = np.sum(d_mul1 * sub_node, axis = 0)
        d_sub1 = d_mul1 * inverse_node
        
        d_std = -d_inverse * (inverse_node ** 2)
        
        d_var = d_std * 0.5 * inverse_node
        
        d_sqr = d_var * np.ones_like(dout) / N
        
        d_sub2 = d_sqr * (2 * sub_node)

        d_sub = d_sub1 + d_sub2
        d_x1 = d_sub
        d_mean = -np.sum(d_sub, axis = 0)
        d_x2 = d_mean * np.ones_like(dout) / N
        
        d_x = d_x1 + d_x2
        
        return d_x, d_gamma, d_beta


    def forward(self, x_train, training=True):
        A = np.asarray(x_train)
        
        self.A_values = [A] # 입력데이터 저장
        self.Z_values = [None]
        self.BN_caches = [None]
        self.activation_values = [None]
        
        # layer 구조 [Z -> batchnorm -> activation]
        for i in range(1, len(self.layer_size)):
            
            # step1: Z값 구하기
            W = self.params[f"W{i}"]
            b = self.params[f"B{i}"]
            
            Z = A @ W + b
            
            # step2: 배치 정규화
            if i < len(self.layer_size) - 1:
                gamma = self.params[f"Gamma{i}"]
                beta = self.params[f"Beta{i}"]
                Z_bn, bn_cache = self.batchnorm_forward(Z, gamma, beta)
                self.BN_caches.append(bn_cache)
                # step3: activation -> dropout
                activation_output = self.activation(Z_bn)
                self.activation_values.append(activation_output)
                A = self.dropout_layers[i].forward(
                    activation_output, training=training
                )
            else:
                self.BN_caches.append(None)
                # 출력층에는 Dropout을 적용 하면 안 됨.
                A = self.activation(Z)
                self.activation_values.append(A)
            
            self.Z_values.append(Z)
            self.A_values.append(A)
        
        self.output_value = A
        return self.output_value
    
    
    def loss(self, x_train, y_train):
        # 0.5 * (predict - y_train)**2
        predict = self.output_value
        
        self.L = np.mean(0.5 * (predict - y_train)**2)
        return self.L
    
    
    def backward(self, y_train):
        
        delta = (self.output_value - y_train) * self.activation_grad(self.output_value)
        
        for i in range(len(self.layer_size)-1, 0, -1):
            A_previous = self.A_values[i - 1]
            
            
            self.grads[f"W{i}"] = A_previous.T @ delta
            self.grads[f"B{i}"] = np.sum(delta, axis = 0, keepdims = True)            
            if i == 1:
                break
            
            W = self.params[f"W{i}"]
            
            # step1: activation
            dA_previous = delta @ (W.T)
            dA_previous = self.dropout_layers[i-1].backward(dA_previous)
            
            act_grad = self.activation_grad(self.activation_values[i-1])
            d_bn = dA_previous * act_grad
            
            bn_cache = self.BN_caches[i-1]
            delta, d_gamma, d_beta = self.batchnorm_backward(d_bn, bn_cache)
            
            self.grads[f"Gamma{i-1}"] = d_gamma
            self.grads[f"Beta{i-1}"] = d_beta
        
    
    def momentum_update(self):
        for key in self.params:
            self.velocity[key] = self.momentum * self.velocity[key] - self.learning_rate * self.grads[key]
            self.params[key] += self.velocity[key]
    
    def sgd_update(self):
        for key in self.params:
            self.params[key] -= self.learning_rate * self.grads[key]

    def adagrad_update(self):
        for key in self.params:
            self.h[key] += self.grads[key] ** 2
            self.params[key] -= (self.learning_rate * self.grads[key]/ (np.sqrt(self.h[key]) + 1e-7))

    def update(self):
        if self.optimizer == "sgd":
            self.sgd_update()
        elif self.optimizer == "momentum":
            self.momentum_update()
        elif self.optimizer == "adagrad":
            self.adagrad_update()
        else:
            raise ValueError(f"error")
    
    def train(self, X_train, Y_train):
        self.forward(X_train)
        self.loss(X_train, Y_train)
        self.backward(Y_train)
        self.update()
        
        return self.L
    
    
    def fit(self, X_train, Y_train, epochs=100):
        loss_record = []
        for i in range(1, epochs + 1):
            loss = self.train(X_train, Y_train)
            loss_record.append(loss)
        return loss_record
    
    def predict(self, X):
        return self.forward(X, training=False)

In [ ]:
# dropout_ratio를 증가시켜가면 학습결과 확인
import time
X = np.array([
    [0, 0, 0],
    [0, 0, 1],
    [0, 1, 0],
    [0, 1, 1],
    [1, 0, 0],
    [1, 0, 1],
    [1, 1, 0],
    [1, 1, 1]
], dtype=np.float64)

y_true = np.array([
    [1, 0],
    [0, 1],
    [0, 1],
    [1, 0],
    [0, 1],
    [1, 0],
    [1, 0],
    [0, 1]
], dtype=np.float64)

for i in range (5):
    above = i * 0.1
    model = NeuralNetwork(layer_size=[3, 4, 2, 2],activation=sigmoid,learning_rate=0.5, dropout_ratio= above)

    start_time = time.perf_counter()
    loss_record = model.fit(X,y_true,epochs=3000)
    y_pred = model.predict(X)

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    print(f"Python 학습 시간: {elapsed_time:.6f}초")
    print("\n학습 결과")

    for sample in range(len(X)):
        print(
            f"{X[sample, 0]:.0f} "
            f"{X[sample, 1]:.0f} "
            f"{X[sample, 2]:.0f} "
            f"-> "
            f"{y_pred[sample, 0]:.6f} "
            f"{y_pred[sample, 1]:.6f}"
        )

Python 학습 시간: 0.695868초

학습 결과
0 0 0 -> 0.989111 0.010995
0 0 1 -> 0.011937 0.987933
0 1 0 -> 0.012378 0.987500
0 1 1 -> 0.980141 0.019772
1 0 0 -> 0.024422 0.975725
1 0 1 -> 0.988398 0.011701
1 1 0 -> 0.989907 0.010210
1 1 1 -> 0.012281 0.987593
Python 학습 시간: 0.560325초

학습 결과
0 0 0 -> 0.987325 0.012678
0 0 1 -> 0.026745 0.973258
0 1 0 -> 0.025987 0.974016
0 1 1 -> 0.986246 0.013757
1 0 0 -> 0.027625 0.972378
1 0 1 -> 0.987072 0.012931
1 1 0 -> 0.987256 0.012747
1 1 1 -> 0.028591 0.971412
Python 학습 시간: 0.568796초

학습 결과
0 0 0 -> 0.997947 0.002053
0 0 1 -> 0.123472 0.876528
0 1 0 -> 0.122411 0.877589
0 1 1 -> 0.997653 0.002347
1 0 0 -> 0.122767 0.877233
1 0 1 -> 0.997711 0.002289
1 1 0 -> 0.997739 0.002261
1 1 1 -> 0.121917 0.878083
Python 학습 시간: 0.543783초

학습 결과
0 0 0 -> 0.848984 0.151016
0 0 1 -> 0.024200 0.975800
0 1 0 -> 0.026649 0.973351
0 1 1 -> 0.848761 0.151239
1 0 0 -> 0.022265 0.977735
1 0 1 -> 0.847709 0.152291
1 1 0 -> 0.848221 0.151779
1 1 1 -> 0.021784 0.978216
Python 학습 시간